## DEMO: Debugging a Broken Agent using Logs

### INTRODUCTION
This demo illustrates how to debug a broken agent using structured logs.  
We’ll build a lightweight agent equipped with basic reasoning, simulate tool usage, and intentionally introduce failures.  

By analyzing the generated log file, you can trace the agent’s decisions, identify errors, and understand how logging enables observability in AI workflows.

-------------------------------------------------------------
OBJECTIVES
-------------------------------------------------------------
- Simulate an agent that performs reasoning steps.  
- Integrate basic tools — a pseudo Calculator and a random tool failure.  
- Introduce intentional bugs to observe failure patterns.  
- Capture reasoning and exceptions using Python’s file-based logging.  
- Use Gradio to interact with the agent in real time.  
- Analyze the log file to identify the root cause of failures.
-------------------------------------------------------------
⚙️ INSTRUCTIONS TO RUN
-------------------------------------------------------------

1️⃣ **Install dependencies:**

In [ ]:
pip install langchain langchain-community langchain-openai gradio wikipedia

2️⃣ Run the script
It will:

- Launch a Gradio interface.
- Log all queries and errors to simple_agent_debug.log.
- Randomly trigger failures for demonstration.

3️⃣ Try these queries:

- calc 25 6

- calc 78*7

- hello

4️⃣ Inspect the log file
After running a few queries, open simple_agent_debug.log.
You’ll find detailed entries including:

- Received queries
- Tool calls and responses
- Exceptions raised
- Execution time per request

In [1]:
# =========================================================
# Simple Demo: Debugging a Broken Agent 
# =========================================================

import gradio as gr
import random
import time
from datetime import datetime

LOG_FILE = "simple_agent_debug.log"

def log_to_file(message: str, error: bool = False):
    """Directly writes to log file immediately (bypasses logging buffers)."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "ERROR" if error else "INFO"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"{timestamp} | {prefix} | {message}\n")
        f.flush()

log_to_file("Starting Simple Agent Debug Demo")

def simple_agent(query: str) -> str:
    """Simulated agent that occasionally breaks."""
    log_to_file(f"Received query: {query}")
    start = time.time()

    try:
        if "calc" in query.lower():
            # Randomly use a broken or valid tool
            tool_name = "BrokenCalculator" if random.random() < 0.4 else "Calculator"
            log_to_file(f"Trying to use tool: {tool_name}")

            if tool_name == "Calculator":
                # Bug: naive parsing fails for cases like "78*7"
                digits = [int(s) for s in query.replace("*", " ").split() if s.isdigit()]
                result = digits[0] * digits[1]
                log_to_file(f"Calculation successful: {result}")
                answer = f"The result is {result}"
            else:
                raise ValueError(f"Tool '{tool_name}' not found")
        else:
            answer = f"I don't know how to handle '{query}'"
            log_to_file("No valid tool found for this query")

    except Exception as e:
        log_to_file(f"Error occurred: {e}", error=True)
        answer = f"Agent failed due to: {e}"

    finally:
        elapsed = time.time() - start
        log_to_file(f"Query completed in {elapsed:.2f}s\n")

    return answer


demo = gr.Interface(
    fn=simple_agent,
    inputs=gr.Textbox(lines=2, placeholder="Try: calc 67 8 or calc 78*7"),
    outputs="text",
    title="Debugging a Broken Agent (Simple Example)",
    description="Agent occasionally fails to find tool or parse input. "
                "Open simple_agent_debug.log after each run to inspect reasoning and errors."
)

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
